In [6]:
import random
import numpy as np

# --- 1. PROBLEM SETUP & PARAMETERS ---
NUM_TASKS = 20      # Number of cloud tasks to schedule
NUM_VMS = 5         # Number of Virtual Machines available
POP_SIZE = 50       # Number of individual schedules in the population
GENERATIONS = 100   # Number of GA iterations
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.1

# Generate a synthetic ETC (Expected Time to Compute) Matrix
# ETC[i][j] represents the execution time of Task i on VM j
np.random.seed(42)
random.seed(42)
ETC = np.random.randint(10, 100, size=(NUM_TASKS, NUM_VMS)).tolist()

# --- 2. CORE GENETIC ALGORITHM FUNCTIONS ---

def initialize_population():
    """
    Creates an initial population where each chromosome represents a schedule.
    A chromosome is a list of length NUM_TASKS.
    The value at index i represents the VM ID assigned to Task i.
    """
    population = []
    for _ in range(POP_SIZE):
        chromosome = [random.randint(0, NUM_VMS - 1) for _ in range(NUM_TASKS)]
        population.append(chromosome)
    return population

def calculate_fitness(chromosome):
    """
    Calculates the fitness of a chromosome.
    Fitness is based on minimizing the Makespan (total execution time).
    We use fitness = 1 / Makespan since GA maximizes fitness.
    """
    vm_load = [0] * NUM_VMS

    # Calculate total workload/execution time on each VM
    for task_id, vm_id in enumerate(chromosome):
        vm_load[vm_id] += ETC[task_id][vm_id]

    makespan = max(vm_load)

    # Avoid division by zero and return fitness
    return 1.0 / makespan if makespan > 0 else 0

def selection(population, fitness_scores):
    """
    Selects two parents using Roulette Wheel Selection.
    """
    total_fitness = sum(fitness_scores)
    # Handle edge case where all fitness scores might be zero
    if total_fitness == 0:
        return random.choice(population), random.choice(population)

    probs = [f / total_fitness for f in fitness_scores]

    parents = random.choices(population, weights=probs, k=2)
    return parents[0], parents[1]

def crossover(parent1, parent2):
    """
    Performs Single-Point Crossover to exchange genetic traits between parents.
    """
    if random.random() < CROSSOVER_RATE:
        point = random.randint(1, NUM_TASKS - 1)
        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]
        return child1, child2
    return parent1.copy(), parent2.copy()

def mutate(chromosome):
    """
    Performs Mutation by reassigning a task to a different VM.
    """
    for task_id in range(NUM_TASKS):
        if random.random() < MUTATION_RATE:
            chromosome[task_id] = random.randint(0, NUM_VMS - 1)
    return chromosome

# --- 3. MAIN GA DRIVER EXTRACTION ---

def run_task_scheduler():
    population = initialize_population()

    best_chromosome = None
    best_makespan = float('inf')

    print("Starting Genetic Algorithm cloud task scheduling...")

    for generation in range(GENERATIONS):
        # Evaluate fitness for all chromosomes
        fitness_scores = [calculate_fitness(chrom) for chrom in population]

        # Track the best schedule in this generation
        for i, fitness in enumerate(fitness_scores):
            makespan = 1.0 / fitness
            if makespan < best_makespan:
                best_makespan = makespan
                best_chromosome = population[i].copy()

        # Generate the next generation
        new_population = []

        # Elitism: carry over the best chromosome automatically
        if best_chromosome:
            new_population.append(best_chromosome.copy())

        while len(new_population) < POP_SIZE:
            parent1, parent2 = selection(population, fitness_scores)
            child1, child2 = crossover(parent1, parent2)

            new_population.append(mutate(child1))
            if len(new_population) < POP_SIZE:
                new_population.append(mutate(child2))

        population = new_population

        # Print progress every 10 generations
        if (generation + 1) % 10 == 0 or generation == 0:
            print(f"Generation {generation + 1:3d}: Minimum Makespan = {best_makespan:.2f} time units")

    # --- 4. OUTPUT RESULTS ---
    print("\n================ FINAL OPTIMAL SCHEDULE ================")
    print(f"Optimal Makespan: {best_makespan:.2f} time units\n")

    # Calculate final individual VM workloads
    final_vm_load = [0] * NUM_VMS
    vm_tasks = {i: [] for i in range(NUM_VMS)}
    for task_id, vm_id in enumerate(best_chromosome):
        final_vm_load[vm_id] += ETC[task_id][vm_id]
        vm_tasks[vm_id].append(f"T{task_id}(time:{ETC[task_id][vm_id]})")

    for vm_id in range(NUM_VMS):
        print(f"VM {vm_id} Total Execution Time: {final_vm_load[vm_id]} units")
        print(f"     Tasks assigned: {', '.join(vm_tasks[vm_id]) if vm_tasks[vm_id] else 'None'}\n")

if __name__ == "__main__":
    run_task_scheduler()


Starting Genetic Algorithm cloud task scheduling...
Generation   1: Minimum Makespan = 239.00 time units
Generation  10: Minimum Makespan = 219.00 time units
Generation  20: Minimum Makespan = 206.00 time units
Generation  30: Minimum Makespan = 198.00 time units
Generation  40: Minimum Makespan = 190.00 time units
Generation  50: Minimum Makespan = 190.00 time units
Generation  60: Minimum Makespan = 173.00 time units
Generation  70: Minimum Makespan = 173.00 time units
Generation  80: Minimum Makespan = 173.00 time units
Generation  90: Minimum Makespan = 173.00 time units
Generation 100: Minimum Makespan = 173.00 time units

================ FINAL OPTIMAL SCHEDULE ================
Optimal Makespan: 173.00 time units

VM 0 Total Execution Time: 137 units
     Tasks assigned: T1(time:92), T2(time:33), T8(time:12)

VM 1 Total Execution Time: 155 units
     Tasks assigned: T3(time:39), T4(time:30), T10(time:18), T12(time:44), T16(time:24)

VM 2 Total Execution Time: 154 units
     Tasks